In [1]:
!git clone https://github.com/Team-TUD/CTAB-GAN-Plus
import sys
sys.path.append('./CTAB-GAN-Plus')
from model.ctabgan import CTABGAN

Cloning into 'CTAB-GAN-Plus'...


In [2]:
from ucimlrepo import fetch_ucirepo

In [3]:
from ucimlrepo import fetch_ucirepo
import numpy as np
import pandas as pd
from sdv.metadata import SingleTableMetadata
import torch
import random

real_estate = fetch_ucirepo(id=477)
X = real_estate.data.features.copy()
y = real_estate.data.targets
print(real_estate.metadata)
print(real_estate.variables)

X = X.drop(columns=["X1 transaction date"], errors="ignore")
if "X4 number of convenience stores" in X.columns:
    X["X4 number of convenience stores"] = pd.to_numeric(
        X["X4 number of convenience stores"], errors="coerce"
    )

data = pd.concat([X, y], axis=1)
target_col = "Y house price of unit area"

# Drop date/time/session/ID features before generator training (high cardinality).
_drop_feature_cols = [
    "Date", "Time", "date_time",
    "session_id", "Session ID", "Session_ID", "session",
    "X1 transaction date",
]
data = data.drop(columns=[c for c in _drop_feature_cols if c in data.columns], errors="ignore")

n_samples = min(1000, len(data))
data = data.sample(n=n_samples, random_state=42).reset_index(drop=True)

for col in data.columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")
    data[col] = data[col].fillna(data[col].median())

processed_data = data.copy()

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(processed_data)

N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

scores = {}
synthetic_datasets = {}
quality_results = []


{'uci_id': 477, 'name': 'Real Estate Valuation', 'repository_url': 'https://archive.ics.uci.edu/dataset/477/real+estate+valuation+data+set', 'data_url': 'https://archive.ics.uci.edu/static/public/477/data.csv', 'abstract': 'The real estate valuation is a regression problem. The market historical data set of real estate valuation are collected from Sindian Dist., New Taipei City, Taiwan. ', 'area': 'Business', 'tasks': ['Regression'], 'characteristics': ['Multivariate'], 'num_instances': 414, 'num_features': 6, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': ['Y house price of unit area'], 'index_col': ['No'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2018, 'last_updated': 'Mon Feb 26 2024', 'dataset_doi': '10.24432/C5J30W', 'creators': ['I-Cheng Yeh'], 'intro_paper': {'ID': 373, 'type': 'NATIVE', 'title': 'Building real estate valuation models with comparative approach through case-based reasoning', 'authors': 'I. Yeh

In [4]:
from sklearn.model_selection import train_test_split
from sdv.evaluation.single_table import evaluate_quality

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

train_real, test_real = train_test_split(
    processed_data,
    test_size=TEST_SIZE,
        random_state=seed
)

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

try:
    data_path = "real_estate_train.csv"
    train_real.to_csv(data_path, index=False)

    ctabgan = CTABGAN(
        raw_csv_path=data_path,
        categorical_columns=[],
        log_columns=[],
        mixed_columns={},
        integer_columns=[],
        problem_type={"Regression": target_col}
    )

    ctabgan.fit()

    synthetic_ctabgan = ctabgan.data_prep.inverse_prep(
        ctabgan.synthesizer.sample(N_SAMPLES)
    )

    # Convert all columns back to numeric where possible
    for col in synthetic_ctabgan.columns:
        synthetic_ctabgan[col] = pd.to_numeric(
            synthetic_ctabgan[col],
            errors="coerce"
        )

        synthetic_ctabgan[col] = synthetic_ctabgan[col].fillna(
            train_real[col].median()
        )

    # Real Estate Valuation quality target is an integer score between 0 and 10
    pass  # keep continuous regression target as-is

    synthetic_datasets["CTABGAN"] = synthetic_ctabgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_ctabgan,
        metadata=train_metadata
    )

    score = quality.get_score()

    scores["CTABGAN"] = score

    print("CTABGAN:", round(score, 4))

except Exception as e:
    print("CTABGAN Failed:", e)



================ SINGLE RUN ================


100%|██████████| 150/150 [03:43<00:00,  1.49s/it]


Finished training in 227.8541111946106  seconds.
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 6/6 [00:00<00:00, 3945.10it/s]|
Column Shapes Score: 90.63%

(2/2) Evaluating Column Pair Trends: |██████████| 15/15 [00:00<00:00, 267.73it/s]|
Column Pair Trends Score: 81.24%

Overall Score (Average): 85.94%

CTABGAN: 0.8594


In [5]:
# WGAN-GP

try:

    import traceback
    from sklearn.preprocessing import StandardScaler
    import torch.nn as nn
    import torch.optim as optim
    from sdv.evaluation.single_table import evaluate_quality

    data_wgan = train_real.copy()

    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(data_wgan)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    real_tensor = torch.tensor(
        scaled_data,
        dtype=torch.float32
    )

    batch_size = 64
    latent_dim = 64
    data_dim = real_tensor.shape[1]

    loader = torch.utils.data.DataLoader(
        real_tensor,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    class Generator(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(latent_dim, 128),
                nn.LayerNorm(128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 256),
                nn.LayerNorm(256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, data_dim)
            )

        def forward(self, z):
            return self.model(z)

    class Critic(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(data_dim, 256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, 128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 1)
            )

        def forward(self, x):
            return self.model(x)

    generator = Generator().to(device)
    critic = Critic().to(device)

    optimizer_G = optim.Adam(
        generator.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    optimizer_C = optim.Adam(
        critic.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    def gradient_penalty(critic, real_samples, fake_samples):

        alpha = torch.rand(real_samples.size(0), 1, device=device)
        alpha = alpha.expand_as(real_samples)

        interpolates = (
            alpha * real_samples +
            (1 - alpha) * fake_samples
        ).requires_grad_(True)

        critic_interpolates = critic(interpolates)

        gradients = torch.autograd.grad(
            outputs=critic_interpolates,
            inputs=interpolates,
            grad_outputs=torch.ones_like(critic_interpolates),
            create_graph=True,
            retain_graph=True
        )[0]

        gradients = gradients.view(gradients.size(0), -1)

        return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

    for epoch in range(100):

        for real_batch in loader:

            real_batch = real_batch.to(device)

            for _ in range(5):

                z = torch.randn(
                    real_batch.size(0),
                    latent_dim,
                    device=device
                )

                fake_batch = generator(z).detach()

                critic_real = critic(real_batch).mean()
                critic_fake = critic(fake_batch).mean()

                gp = gradient_penalty(
                    critic,
                    real_batch,
                    fake_batch
                )

                critic_loss = (
                    critic_fake
                    - critic_real
                    + 10 * gp
                )

                optimizer_C.zero_grad()
                critic_loss.backward()
                optimizer_C.step()

            z = torch.randn(
                real_batch.size(0),
                latent_dim,
                device=device
            )

            fake = generator(z)

            generator_loss = -critic(fake).mean()

            optimizer_G.zero_grad()
            generator_loss.backward()
            optimizer_G.step()

    generator.eval()

    with torch.no_grad():

        z = torch.randn(
            N_SAMPLES,
            latent_dim,
            device=device
        )

        synthetic_scaled = generator(z).cpu().numpy()

    synthetic = scaler.inverse_transform(synthetic_scaled)

    synthetic_wgan = pd.DataFrame(
        synthetic,
        columns=data_wgan.columns
    )

    synthetic_datasets["WGAN_GP"] = synthetic_wgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_wgan,
        metadata=train_metadata
    )

    scores["WGAN_GP"] = quality.get_score()

    print("WGAN_GP:", round(scores["WGAN_GP"], 4))

    del generator
    del critic
    del real_tensor

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("WGAN_GP Failed:")
    traceback.print_exc()

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 6/6 [00:00<00:00, 356.66it/s]|
Column Shapes Score: 79.74%

(2/2) Evaluating Column Pair Trends: |██████████| 15/15 [00:00<00:00, 281.38it/s]|
Column Pair Trends Score: 96.45%

Overall Score (Average): 88.1%

WGAN_GP: 0.881


In [6]:
# SDV MODELS

from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)

from sdv.evaluation.single_table import evaluate_quality

sdv_models = {
    "CTGAN": CTGANSynthesizer(metadata=train_metadata),
    "CopulaGAN": CopulaGANSynthesizer(metadata=train_metadata),
    "TVAE": TVAESynthesizer(metadata=train_metadata),
    "GaussianCopula": GaussianCopulaSynthesizer(metadata=train_metadata)
}

for model_name, model in sdv_models.items():

    try:

        model.fit(train_real)

        synthetic_data = model.sample(N_SAMPLES)

        pass  # keep continuous regression target as-is

        synthetic_datasets[model_name] = synthetic_data.copy()

        quality = evaluate_quality(
            real_data=train_real,
            synthetic_data=synthetic_data,
            metadata=train_metadata
        )

        scores[model_name] = quality.get_score()

        print(
            f"{model_name}: {round(scores[model_name], 4)}"
        )

    except Exception as e:

        print(
            f"{model_name} Failed: {e}"
        )


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 6/6 [00:00<00:00, 355.97it/s]|
Column Shapes Score: 81.2%

(2/2) Evaluating Column Pair Trends: |██████████| 15/15 [00:00<00:00, 433.67it/s]|
Column Pair Trends Score: 68.88%

Overall Score (Average): 75.04%

CTGAN: 0.7504
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 6/6 [00:00<00:00, 348.29it/s]|
Column Shapes Score: 73.0%

(2/2) Evaluating Column Pair Trends: |██████████| 15/15 [00:00<00:00, 334.65it/s]|
Column Pair Trends Score: 72.1%

Overall Score (Average): 72.55%

CopulaGAN: 0.7255
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 6/6 [00:00<00:00, 1820.05it/s]|
Column Shapes Score: 89.3%

(2/2) Evaluating Column Pair Trends: |██████████| 15/15 [00:00<00:00, 299.05it/s]|
Column Pair Trends Score: 97.77%

Overall Score (Average): 93.54%

TVAE: 0.9354
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 6/6 [00:00<00:00, 749.90it/s]|
Column Shapes Score: 83.3

In [7]:
from sklearn.base import clone
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor

regressors = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.001, max_iter=5000),
    'ElasticNet': ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000),
    'SVR_RBF': SVR(kernel='rbf', C=1.0, epsilon=0.1),
    'KNN': KNeighborsRegressor(n_neighbors=5),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'GradientBoost': GradientBoostingRegressor(random_state=42),
}

EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
GENERATORS_TO_EVAL = list(synthetic_datasets.keys())

print(f'Regression evaluation: {len(regressors)} models, {len(EVAL_SEEDS)} seeds, {len(GENERATORS_TO_EVAL)} generators')


Regression evaluation: 10 models, 10 seeds, 6 generators


In [8]:
from sklearn.model_selection import train_test_split

def evaluate_regression_models(train_df, test_df, label_col, models, test_size=0.2, seeds=None, use_holdout=False, schema_df=None):
    if seeds is None:
        seeds = EVAL_SEEDS
    if schema_df is None:
        schema_df = test_df if use_holdout else train_df

    train_df = align_to_train_schema(train_df, schema_df, label_col)
    test_df = align_to_train_schema(test_df, schema_df, label_col)
    results = []

    for name, model in models.items():
        r2_scores = []
        mse_scores = []
        rmse_scores = []
        mae_scores = []

        for seed in seeds:
            X_train = train_df.drop(columns=[label_col])
            y_train = train_df[label_col]

            X_test = test_df.drop(columns=[label_col])
            y_test = test_df[label_col]

            if not use_holdout:
                X_train, _, y_train, _ = train_test_split(
                    X_train, y_train, test_size=test_size, random_state=seed
                )
                _, X_test, _, y_test = train_test_split(
                    X_test, y_test, test_size=test_size, random_state=seed
                )

            reg = clone(model)
            if 'random_state' in reg.get_params():
                reg.set_params(random_state=seed)

            reg.fit(X_train, y_train)
            y_pred = reg.predict(X_test)

            mse = mean_squared_error(y_test, y_pred)
            r2_scores.append(r2_score(y_test, y_pred))
            mse_scores.append(mse)
            rmse_scores.append(np.sqrt(mse))
            mae_scores.append(mean_absolute_error(y_test, y_pred))

        results.append({
            'Model': name,
            'R2 Mean': np.mean(r2_scores),
            'R2 Std': np.std(r2_scores),
            'MSE Mean': np.mean(mse_scores),
            'MSE Std': np.std(mse_scores),
            'RMSE Mean': np.mean(rmse_scores),
            'RMSE Std': np.std(rmse_scores),
            'MAE Mean': np.mean(mae_scores),
            'MAE Std': np.std(mae_scores),
            'R2 (Mean±Std)': f"{np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}",
            'MSE (Mean±Std)': f"{np.mean(mse_scores):.4f} ± {np.std(mse_scores):.4f}",
            'RMSE (Mean±Std)': f"{np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}",
            'MAE (Mean±Std)': f"{np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}",
        })

    return pd.DataFrame(results).sort_values(by='R2 Mean', ascending=False)


In [9]:
def align_to_train_schema(df_to_align, schema_reference_df, label_col):
    feature_cols_schema = [col for col in schema_reference_df.columns if col != label_col]

    for col in feature_cols_schema:
        if col not in df_to_align.columns:
            df_to_align[col] = schema_reference_df[col].median()

    extra_cols = [col for col in df_to_align.columns if col not in schema_reference_df.columns]
    if extra_cols:
        df_to_align = df_to_align.drop(columns=extra_cols)

    final_ordered_cols = [col for col in schema_reference_df.columns if col in df_to_align.columns]
    return df_to_align[final_ordered_cols]

print('TRTR (Train Real, Test Real) — repeated 80/20 splits per seed')
trtr_results = evaluate_regression_models(
    train_df=processed_data,
    test_df=processed_data,
    label_col=target_col,
    models=regressors,
    seeds=EVAL_SEEDS,
    use_holdout=False,
    schema_df=processed_data,
)
display(trtr_results[['Model', 'R2 (Mean±Std)', 'MSE (Mean±Std)', 'RMSE (Mean±Std)', 'MAE (Mean±Std)']])

all_comparisons = []
for synth_name in GENERATORS_TO_EVAL:
    if synth_name not in synthetic_datasets:
        print(f'Skipping {synth_name} - no synthetic dataset')
        continue

    print(f'{synth_name} - TSTR (train on synthetic, test on real — split per seed)')
    tstr_results = evaluate_regression_models(
        train_df=synthetic_datasets[synth_name],
        test_df=test_real,
        label_col=target_col,
        models=regressors,
        seeds=EVAL_SEEDS,
        use_holdout=False,
        schema_df=train_real,
    )
    display(tstr_results[['Model', 'R2 (Mean±Std)', 'MSE (Mean±Std)', 'RMSE (Mean±Std)', 'MAE (Mean±Std)']])

    comparison = trtr_results.merge(tstr_results, on='Model', suffixes=('_TRTR', '_TSTR'))
    comparison['R2_Drop'] = comparison['R2 Mean_TRTR'] - comparison['R2 Mean_TSTR']
    comparison['MSE_Increase'] = comparison['MSE Mean_TSTR'] - comparison['MSE Mean_TRTR']
    comparison['RMSE_Increase'] = comparison['RMSE Mean_TSTR'] - comparison['RMSE Mean_TRTR']
    comparison['MAE_Increase'] = comparison['MAE Mean_TSTR'] - comparison['MAE Mean_TRTR']
    comparison['Synthetic_Model'] = synth_name
    all_comparisons.append(comparison)

combined_comparison = pd.concat(all_comparisons, ignore_index=True)
summary = (
    combined_comparison
    .groupby('Synthetic_Model', as_index=False)[['R2_Drop', 'MSE_Increase', 'RMSE_Increase', 'MAE_Increase']]
    .mean()
    .sort_values('R2_Drop')
)

display(summary)


TRTR (Train Real, Test Real) — repeated 80/20 splits per seed


,Model,R2 (Mean±Std),MSE (Mean±Std),RMSE (Mean±Std),MAE (Mean±Std)
9,GradientBoost,0.6749 ± 0.1043,66.3633 ± 30.5495,7.9369 ± 1.8354,5.1871 ± 0.3941
7,RandomForest,0.6671 ± 0.1199,68.4771 ± 34.3396,8.0159 ± 2.0550,5.0846 ± 0.5895
8,ExtraTrees,0.6341 ± 0.1293,74.7122 ± 36.1048,8.3969 ± 2.0504,5.3160 ± 0.5733
5,KNN,0.6118 ± 0.0813,78.2000 ± 27.8060,8.7127 ± 1.5127,5.9095 ± 0.5526
2,Lasso,0.5335 ± 0.0907,93.7787 ± 32.0323,9.5532 ± 1.5858,6.5800 ± 0.4228
0,LinearRegression,0.5317 ± 0.0910,94.1356 ± 32.1274,9.5716 ± 1.5875,6.5858 ± 0.4312
3,ElasticNet,0.5310 ± 0.0905,94.1519 ± 31.6940,9.5753 ± 1.5700,6.6955 ± 0.3690
4,SVR_RBF,0.5301 ± 0.0757,93.9362 ± 28.5491,9.5869 ± 1.4240,6.5782 ± 0.4521
1,Ridge,0.5261 ± 0.0902,95.0663 ± 31.6136,9.6247 ± 1.5595,6.7396 ± 0.3624
6,DecisionTree,0.4838 ± 0.1311,101.3645 ± 29.4744,9.9656 ± 1.4322,6.2703 ± 0.2313


CTABGAN - TSTR (train on synthetic, test on real — split per seed)


,Model,R2 (Mean±Std),MSE (Mean±Std),RMSE (Mean±Std),MAE (Mean±Std)
3,ElasticNet,0.5654 ± 0.1330,81.6462 ± 34.9271,8.8437 ± 1.8535,6.7732 ± 1.5073
1,Ridge,0.5547 ± 0.1351,83.6987 ± 35.9082,8.9568 ± 1.8639,6.9783 ± 1.5175
2,Lasso,0.5315 ± 0.1255,87.0040 ± 30.5510,9.1763 ± 1.6729,6.8745 ± 1.2057
0,LinearRegression,0.5280 ± 0.1261,87.5814 ± 30.4222,9.2093 ± 1.6646,6.9097 ± 1.1917
8,ExtraTrees,0.4716 ± 0.1624,99.3310 ± 43.7643,9.7530 ± 2.0520,7.4863 ± 1.5506
9,GradientBoost,0.4704 ± 0.1151,99.1282 ± 34.8946,9.8161 ± 1.6652,7.6146 ± 1.0944
7,RandomForest,0.4359 ± 0.1081,107.2946 ± 43.5900,10.1815 ± 1.9055,7.7686 ± 1.3336
4,SVR_RBF,0.3634 ± 0.0806,121.2264 ± 45.9896,10.8473 ± 1.8874,8.6756 ± 1.5304
5,KNN,0.1857 ± 0.2879,143.8353 ± 36.2859,11.9037 ± 1.4619,9.6430 ± 1.4556
6,DecisionTree,-0.5800 ± 0.5667,280.7276 ± 90.3296,16.5055 ± 2.8800,13.2519 ± 2.2026


WGAN_GP - TSTR (train on synthetic, test on real — split per seed)


...

TVAE - TSTR (train on synthetic, test on real — split per seed)


,Model,R2 (Mean±Std),MSE (Mean±Std),RMSE (Mean±Std),MAE (Mean±Std)
9,GradientBoost,0.6128 ± 0.1423,69.5521 ± 20.7321,8.2435 ± 1.2636,6.5110 ± 1.1196
7,RandomForest,0.6110 ± 0.0898,71.0819 ± 17.8475,8.3698 ± 1.0144,6.7458 ± 0.9589
0,LinearRegression,0.5594 ± 0.2055,80.5617 ± 37.6425,8.6847 ± 2.2667,6.7863 ± 1.4810
2,Lasso,0.5594 ± 0.2055,80.5665 ± 37.6446,8.6850 ± 2.2667,6.7865 ± 1.4810
3,ElasticNet,0.5594 ± 0.2055,80.5672 ± 37.6447,8.6850 ± 2.2667,6.7865 ± 1.4810
1,Ridge,0.5594 ± 0.2055,80.5695 ± 37.6455,8.6851 ± 2.2667,6.7866 ± 1.4810
8,ExtraTrees,0.5253 ± 0.1196,86.1631 ± 21.5585,9.2164 ± 1.1052,7.3077 ± 1.0477
5,KNN,0.4659 ± 0.1193,97.7818 ± 25.4148,9.8097 ± 1.2457,7.9524 ± 0.7546
4,SVR_RBF,0.3775 ± 0.1031,115.1286 ± 32.2892,10.6423 ± 1.3673,8.2774 ± 1.1706
6,DecisionTree,0.1075 ± 0.2976,157.1892 ± 38.9325,12.4381 ± 1.5757,10.0853 ± 1.6587


GaussianCopula - TSTR (train on synthetic, test on real — split per seed)


,Model,R2 (Mean±Std),MSE (Mean±Std),RMSE (Mean±Std),MAE (Mean±Std)
9,GradientBoost,0.6199 ± 0.1755,67.0133 ± 20.9526,8.0560 ± 1.4542,6.2221 ± 1.2308
7,RandomForest,0.5981 ± 0.1718,72.7228 ± 27.1123,8.3710 ± 1.6275,6.3240 ± 1.4028
4,SVR_RBF,0.5858 ± 0.1395,77.0465 ± 31.7561,8.6049 ± 1.7329,6.6584 ± 1.3171
8,ExtraTrees,0.5849 ± 0.1418,74.9327 ± 22.0966,8.5574 ± 1.3055,6.4336 ± 1.0923
0,LinearRegression,0.5616 ± 0.1782,77.8611 ± 23.1756,8.7177 ± 1.3648,7.1621 ± 1.1375
3,ElasticNet,0.5616 ± 0.1782,77.8623 ± 23.1760,8.7178 ± 1.3648,7.1622 ± 1.1375
1,Ridge,0.5616 ± 0.1782,77.8624 ± 23.1761,8.7178 ± 1.3648,7.1622 ± 1.1376
2,Lasso,0.5616 ± 0.1782,77.8624 ± 23.1761,8.7178 ± 1.3648,7.1622 ± 1.1375
5,KNN,0.3709 ± 0.2495,113.9222 ± 41.2736,10.4946 ± 1.9456,8.0799 ± 1.4596
6,DecisionTree,0.1616 ± 0.3080,153.4705 ± 60.4220,12.1501 ± 2.4176,9.6459 ± 1.7033


,Synthetic_Model,R2_Drop,MSE_Increase,RMSE_Increase,MAE_Increase
3,GaussianCopula,0.055658,1.037034,0.016527,1.106586
4,TVAE,0.078634,5.897584,0.251983,1.307881
5,WGAN_GP,0.090541,7.070880,0.420089,1.696712
0,CTABGAN,0.219740,33.128753,1.425348,2.102906
2,CopulaGAN,0.982789,173.417123,6.447162,6.834445
1,CTGAN,1.814857,322.505682,10.776679,10.423761


In [10]:
quality_df = pd.DataFrame.from_dict(scores, orient='index', columns=['Quality Score'])
quality_df.index.name = 'Synthetic Model'
quality_df = quality_df.reset_index()

output_file = 'TRTR_TSTR_results_real_estate.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    quality_df.to_excel(writer, sheet_name='Quality_Metrics', index=False)
    trtr_results.to_excel(writer, sheet_name='TRTR_Results', index=False)
    combined_comparison.to_excel(writer, sheet_name='All_Comparisons', index=False)
    summary.to_excel(writer, sheet_name='Summary', index=False)
    for synth_name in GENERATORS_TO_EVAL:
        if synth_name in combined_comparison['Synthetic_Model'].values:
            synth_results = combined_comparison[combined_comparison['Synthetic_Model'] == synth_name]
            synth_results.to_excel(writer, sheet_name=synth_name[:31], index=False)

print(f'Results saved to: {output_file}')


Results saved to: TRTR_TSTR_results_real_estate.xlsx
